## Máster en Big Data y Data Science

### Metodologías de gestión y diseño de proyectos de big data

#### AP1 - Limpieza de los datos

---

En esta libreta se realiza una evaluación básica de calidad de los datos del escenario y se ejecutan acciones de limpieza.

In [19]:
# Se importan las librerías necesarias y se suprimen las advertencias
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore',category=FutureWarning)
warnings.filterwarnings('ignore',category=UserWarning)

Lectura del mismo dataset inicial

In [20]:
# Lectura de los datos
df_clientes = pd.read_csv("../data/raw/datos_clientes.csv", sep=";")
display(df_clientes.head(5))

df_resenas = pd.read_csv("../data/raw/datos_resenas.csv", sep=";")
display(df_resenas.head(5))

,id_cliente,edad,genero,ciudad,antiguedad_cliente_meses,canal_registro
0,1,42.0,Masculino,Buenos Aires,68,Tienda física
1,2,24.0,Otro,Mendoza,84,Tienda física
2,3,48.0,Femenino,Rosario,47,Online
3,4,50.0,Masculino,Rosario,61,Tienda física
4,5,18.0,Masculino,San Miguel de Tucumán,58,Online


,id_resena,id_cliente,categoria_producto,canal_compra,plataforma,calificacion,texto_resena,tiempo_respuesta_horas,problema_resuelto,reclamo_formal,importe_compra,fecha_resena,sentimiento
0,1,294,Deportes,Online,Flipkart,4,El envío fue rápido y el colchoneta llegó en p...,4.9,Sí,No,22738.67,2025-03-29,Positivo
1,2,2376,Calzado,Online,Flipkart,4,"El sandalias es tal cual la descripción, estoy...",5.1,Sí,No,42589.72,2025-09-15,Positivo
2,3,1670,Electrónica,Online,Meesho,5,"Muy conforme con la compra, la calidad del par...",4.3,Sí,No,29640.40,2025-07-22,Positivo
3,4,310,Hogar,Online,Falabella,3,Tuve varios problemas con el lámpara y el sopo...,6.7,Sí,No,28231.19,2025-07-06,Neutral
4,5,710,Deportes,Offline,Zepto,5,"Excelente botella deportiva, superó mis expect...",12.5,Sí,No,14683.00,2025-09-27,Positivo


##### Transformaciones sobre los datos

Verificación de nulos

In [21]:
# Se verifica la cantidad de valores nulos en cada columna
print("Valores nulos en el dataset de clientes:")
for col in df_clientes.columns:
   print(f"Atributo: {col} - Valores nulos: {df_clientes[col].isna().sum()} ({df_clientes[col].isna().sum()/len(df_clientes)*100:.2f}%)")

Valores nulos en el dataset de clientes:
Atributo: id_cliente - Valores nulos: 0 (0.00%)
Atributo: edad - Valores nulos: 243 (2.34%)
Atributo: genero - Valores nulos: 0 (0.00%)
Atributo: ciudad - Valores nulos: 0 (0.00%)
Atributo: antiguedad_cliente_meses - Valores nulos: 0 (0.00%)
Atributo: canal_registro - Valores nulos: 0 (0.00%)


In [22]:
# Se verifica la cantidad de valores nulos en cada columna
print("Valores nulos en el dataset de reseñas:")
for col in df_resenas.columns:
   print(f"Atributo: {col} - Valores nulos: {df_resenas[col].isna().sum()} ({df_resenas[col].isna().sum()/len(df_resenas)*100:.2f}%)")

Valores nulos en el dataset de reseñas:
Atributo: id_resena - Valores nulos: 0 (0.00%)
Atributo: id_cliente - Valores nulos: 0 (0.00%)
Atributo: categoria_producto - Valores nulos: 0 (0.00%)
Atributo: canal_compra - Valores nulos: 0 (0.00%)
Atributo: plataforma - Valores nulos: 0 (0.00%)
Atributo: calificacion - Valores nulos: 0 (0.00%)
Atributo: texto_resena - Valores nulos: 0 (0.00%)
Atributo: tiempo_respuesta_horas - Valores nulos: 757 (3.02%)
Atributo: problema_resuelto - Valores nulos: 0 (0.00%)
Atributo: reclamo_formal - Valores nulos: 0 (0.00%)
Atributo: importe_compra - Valores nulos: 0 (0.00%)
Atributo: fecha_resena - Valores nulos: 0 (0.00%)
Atributo: sentimiento - Valores nulos: 0 (0.00%)


Selección de datos (filtro de columnas)

In [23]:
# Se establece qué columnas se eliminan
# 'canal_registro' no aporta información relevante para el escenario de análisis de sentimiento
col_eliminar_clientes = ['canal_registro']
col_eliminar_resenas = []

# Se ejecuta la operación
df_clientes.drop(col_eliminar_clientes, inplace=True, axis=1)
df_resenas.drop(col_eliminar_resenas, inplace=True, axis=1)

Limpieza de datos (filtro de filas)

Se descartan edades no realistas (mayores a 90 años) y se eliminan reseñas duplicadas.

In [24]:
# Se filtran los datos para eliminar registros con edades superiores a 90 años
df_clientes_filtrado = df_clientes.copy()
df_clientes_filtrado = df_clientes_filtrado[(df_clientes_filtrado['edad'] < 90) | (df_clientes_filtrado['edad'].isna())]
print(f"Registros eliminados por edad superior a 90 años: {len(df_clientes) - len(df_clientes_filtrado)}")

# Se eliminan reseñas duplicadas
duplicados_resenas = df_resenas.duplicated().sum()
print(f"Reseñas duplicadas encontradas: {duplicados_resenas}")
df_resenas_filtrado = df_resenas.drop_duplicates().copy()

Registros eliminados por edad superior a 90 años: 28
Reseñas duplicadas encontradas: 63


Imputación de valores para los campos edad y tiempo_respuesta_horas según mediana del dataset

In [25]:
# Tratamiento de valores nulos para edad (se imputa con la mediana según ciudad)
df_clientes_filtrado['edad'] = df_clientes_filtrado.groupby("ciudad")["edad"]\
                       .transform(lambda x: x.fillna(x.median()))

# Tratamiento de nulos para tiempo_respuesta_horas (se imputa con la mediana según categoría de producto)
df_resenas_filtrado['tiempo_respuesta_horas'] = df_resenas_filtrado.groupby("categoria_producto")["tiempo_respuesta_horas"]\
                       .transform(lambda x: x.fillna(x.median()))

Integración de ambos datasets en uno solo

In [26]:
df_integrado = pd.merge(df_resenas_filtrado, df_clientes_filtrado, on='id_cliente', how='inner')

print(f"Filas del dataset integrado con los filtros realizados: {df_integrado.shape[0]}")
print(f"Columnas del dataset integrado post integración: {df_integrado.shape[1]}")
print(f'El dataset original tenía {df_resenas.shape[0]} filas en Reseñas y {df_clientes.shape[0]} filas en Clientes.')
print(f'Se han eliminado {df_resenas.shape[0] - df_resenas_filtrado.shape[0]} filas de reseñas en total.')

Filas del dataset integrado con los filtros realizados: 24934
Columnas del dataset integrado post integración: 17
El dataset original tenía 25063 filas en Reseñas y 10400 filas en Clientes.
Se han eliminado 63 filas de reseñas en total.


Generación de atributos

In [27]:
# Longitud del texto de la reseña, en caracteres
df_integrado["longitud_resena"] = df_integrado["texto_resena"].str.len()

# Cantidad de palabras del texto de la reseña
df_integrado["cantidad_palabras_resena"] = df_integrado["texto_resena"].str.split().apply(len)

# Tiempo de respuesta expresado en días, más interpretable que en horas
df_integrado["tiempo_respuesta_dias"] = df_integrado["tiempo_respuesta_horas"] / 24

##########################################################################
# TODO: otros atributos que podrían agregarse:
# - cantidad de reseñas previas del mismo cliente
# - gasto promedio por categoría de producto del cliente
# - proporción de reclamos formales sobre el total de compras del cliente
##########################################################################

Transformaciones sobre atributos

In [28]:
# Se eliminan columnas identificadoras, la fecha (no se utiliza en esta etapa)
# y 'calificacion', dado que determina directamente a 'sentimiento' y generaría data leakage
columnas_a_eliminar = [
    "id_cliente",
    "id_resena",
    "calificacion",
    "fecha_resena",
]

df_integrado.drop(columnas_a_eliminar, inplace=True, axis=1)

Validaciones finales

In [29]:
# Se verifica la cantidad de valores nulos en cada columna del dataset integrado
print("Valores nulos en el dataset integrado:")
for col in df_integrado.columns:
   print(f"Atributo: {col} - Valores nulos: {df_integrado[col].isna().sum()} ({df_integrado[col].isna().sum()/len(df_integrado)*100:.2f}%)")

# Se verifica la presencia de valores duplicados en el dataset integrado
num_duplicados = df_integrado.duplicated().sum()
print(f"\nNúmero de filas duplicadas en el dataset integrado: {num_duplicados}")

Valores nulos en el dataset integrado:
Atributo: categoria_producto - Valores nulos: 0 (0.00%)
Atributo: canal_compra - Valores nulos: 0 (0.00%)
Atributo: plataforma - Valores nulos: 0 (0.00%)
Atributo: texto_resena - Valores nulos: 0 (0.00%)
Atributo: tiempo_respuesta_horas - Valores nulos: 0 (0.00%)
Atributo: problema_resuelto - Valores nulos: 0 (0.00%)
Atributo: reclamo_formal - Valores nulos: 0 (0.00%)
Atributo: importe_compra - Valores nulos: 0 (0.00%)
Atributo: sentimiento - Valores nulos: 0 (0.00%)
Atributo: edad - Valores nulos: 0 (0.00%)
Atributo: genero - Valores nulos: 0 (0.00%)
Atributo: ciudad - Valores nulos: 0 (0.00%)
Atributo: antiguedad_cliente_meses - Valores nulos: 0 (0.00%)
Atributo: longitud_resena - Valores nulos: 0 (0.00%)
Atributo: cantidad_palabras_resena - Valores nulos: 0 (0.00%)
Atributo: tiempo_respuesta_dias - Valores nulos: 0 (0.00%)

Número de filas duplicadas en el dataset integrado: 0


In [30]:
resumen = pd.DataFrame({
    "Atributo": df_integrado.columns,
    "Tipo de Dato": df_integrado.dtypes,
    "Valores Únicos": df_integrado.nunique(),
    "Valores Nulos": df_integrado.isna().sum(),
})

resumen

,Atributo,Tipo de Dato,Valores Únicos,Valores Nulos
categoria_producto,categoria_producto,object,8,0
canal_compra,canal_compra,object,2,0
plataforma,plataforma,object,8,0
texto_resena,texto_resena,object,3126,0
tiempo_respuesta_horas,tiempo_respuesta_horas,float64,657,0
problema_resuelto,problema_resuelto,object,2,0
reclamo_formal,reclamo_formal,object,2,0
importe_compra,importe_compra,float64,24888,0
sentimiento,sentimiento,object,3,0
edad,edad,float64,60,0


----

Exportación del dataset unificado a un archivo nuevo (en el directorio de procesados)

In [31]:
# Exportar el DataFrame limpio a un nuevo archivo CSV
df_integrado.to_csv('../data/processed/datos_integrados.csv', index=False)